### Init

In [0]:
import  pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim, col, length

In [0]:
#Data Dictionary
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

### Reading Bronze data - prd_info

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

 #Silver Data transformations

 ##Trimming 

In [0]:
for item in df.schema.fields:
    if isinstance(item.dataType, StringType):
        df = df.withColumn(item.name, trim(col(item.name)))



 ##Product Key Parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring (col("prd_key"), 7, length(col("prd_key")) - 6)
)

 ##Cost cleanup

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

 #Product Normalization

In [0]:
   # Normalize product line
df = (
    df
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

 #Date casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

In [0]:
df = df.select([F.col(item).alias(RENAME_MAP[item]) for item in df.columns])

 ## Writing to the Silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")